#**Train dữ liệu YOLOv12 với dữ liệu tự cá kiếm**

**Đây là bước chuẩn bị, trong này là bước kiểm tra xem máy tính có đang sử dụng card NVIDIA để training AI, nếu là GoogleColab có thể bỏ qua bước này, nhưng tốt nhất vẫn là local laptop vì cái Colab kia nó bị giới hạn, nếu máy có card NVIDIA mà không hiển thị gì thì cài CUDA và CUDNN phù hợp với NVIDIA đang dùng, cách cài có trên Youtube, không làm được nữa thì phải chịu. Còn nếu Laptop dùng card AMD thì tìm hiểu về cái ZLUDA, laptop intel thì bỏ tiền ra mua Colab nhé**

In [2]:
!nvidia-smi

/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
Mon Feb  9 22:17:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.57                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   52C    P0             18W /   72W |       0MiB /   6144MiB |      0%      Default |
|                                         |                       

Lưu ý: Muốn cài torch có thể chạy được gpu thì hãy tra trên Google Pytorch CUDA, tuyệt đối không dùng "pip install torch" vì nó sẽ dùng CPU thay vì GPU. Tại sao tui muốn dùng GPU thì phần sau sẽ giải thích sau.

In [3]:
import torch
print(torch.__version__)
print(torch.version.cuda)




2.10.0+cu128
12.8


**Bước 01 # Cài thư viện Ultralytics: Trong này có YOLOv luôn. Cài rồi thì bỏ qua**

In [4]:
!pip install ultralytics

/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


**Bước 02 # Import các thư viện cần thiết, đảm bảo dòng Ultralytics kết quả có CUDA**

In [5]:
import os
import ultralytics
ultralytics.checks()

Ultralytics 8.4.13 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
Setup complete ✅ (12 CPUs, 15.5 GB RAM, 61.9/1006.9 GB disk)


In [6]:
from ultralytics import YOLO
from IPython.display import Image

In [7]:
HOME = os.getcwd()
print(HOME)

/mnt/e/Complete/ComputerVision


**Bước 03 # Tải Dataset từ Roboflow đã train trước đó, kể cả đã tải rồi vẫn phải bấm để nó lấy các biến môi trường**

In [8]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="b0eCGuE4vmmcYDSAxchp")
project = rf.workspace("nguyenkhoa").project("animaldetection-j5ew3")
version = project.version(14)
dataset = version.download("yolov12")


/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
loading Roboflow workspace...
loading Roboflow project...


**Bước 04 # Bắt đầu huấn luyện mô hình**

Đầu tiên, cần phải thiết lập lại file `data.yaml` trong file dataset mới được tải về, với mục tiêu là có thể huấn luyện được với YOLOv12. Đoạn mã dưới là dành cho hệ điều hành Windows, còn với Linux hoặc GoogleColab thì hãy mở file đó lên, xóa 4 dòng cuối cùng, copy 3 dòng đầu tiên rồi dán xuống là okêla

In [9]:
import os

yaml_path = os.path.join(dataset.location, "data.yaml")
with open(yaml_path, "r") as f:
    lines = f.readlines()
lines = lines[:-4]
lines += [
    "test: ../test/images\n",
    "train: ../train/images\n",
    "val: ../valid/images\n"
]
with open(yaml_path, "w") as f:
    f.writelines(lines)

Được rồi, bắt đầu việc huấn luyện nào. Với việc đã thiết lập các biến môi trường ở bước số 3, với máy có 12gb VRAM, hãy sử dụng `yolov12m.yaml`, nhưng nếu máy không có quá nhiều VRAM (hoặc là không có, hoặc là có A100), hãy thử một model có tham số lớn hơn hoặc n hơn như  `yolov12n.pt`, `yolov12s.pt`, `yolov12l.pt`, `yolov12x.pt`), hãy lượng sức mình, không tham lam, thời gian là vàng. Ở đây đã set với tham số epochs là 40. Có thể set lại một số tham số tùy chỉnh dựa vào các tài nguyên có sẵn như batch size (Số lượng ảnh đầu vào xử lý cùng một lúc), image size (Độ phân giải ảnh), và các thông số khác như:
**scale:**: Phóng to hoặc thu nhỏ ảnh, hệ thống có thể học được các đối tượng theo nhiều kiểu khác nhau, nhưng cần máy lực
**mosaic**: Ghép 4 hình ảnh khác nhau thành một hình ảnh duy nhất, tăng tính đa dạng dữ liệu, nhược điểm như trên
**mixup**: Trộn hai hình ảnh và nhãn tương ứng bằng cách lấy trung bình có trọng số. Tác dụng làm mượt ranh giới quyết định của mô hình, giảm overfitting. Nhược: Có thể làm giảm độ sắc nét của hình ảnh, chỉ dùng vừa phải.
**copy-paste**: Cắt đối tượng từ một hình ảnh và dán vào hình ảnh khác. Tăng số lượng đối tượng trong ảnh, cải thiện khả năng phát hiện vật thể nhỏ hoặc hiếm, chỉ nên dùng khi cái dataset bị mất cân bằng dữ liệu với các mẫu dữ liệu

**Chú ý:** **Sau mỗi epochs đều có một file last.pt, nếu cần train tiếp, hãy thay model = YOLO("đường dẫn đến file last.pt")**
**Chú ý:** **Quan trọng nói 3 lần, hãy train bằng GPU NVIDIA, nếu bỏ qua cái này và train bằng CPU, thì đó là một nỗ lực không thể coi thường**

In [10]:
model = YOLO('yolo12s.pt')

In [11]:
results = model.train(data=f'{dataset.location}/data.yaml', epochs=100)

Ultralytics 8.4.13 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/e/Complete/ComputerVision/animaldetection-14/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train5, nbs=64, nms=False, opset=None, optimize=False, opti

KeyboardInterrupt: 

**Bước # 05 Xem kết quả được dùng để cho vào trong báo cáo**
Phần này dùng để báo cáo, tự tìm hiểu

In [ ]:
Image(filename=f'{HOME}/runs/detect/train/confusion_matrix.png', width=1000)

In [ ]:
Image(filename=f'{HOME}/runs/detect/train4/confusion_matrix_normalized.png', width=1000)

In [ ]:
Image(filename=f'{HOME}/runs/detect/train4/results.png', width=1000)

Precision = TP / (TP + FP)

Precision là biến xác định độ chính xác của mô h


In [ ]:
Image(filename=f'{HOME}/runs/detect/train4/BoxP_curve.png', width=1000)

Recall = TP / (TP + FN)

Recall (hay còn gọi là Sensitivity hoặc True Positive Rate) đo lường khả năng mô hình phát hiện đúng các trường hợp dương tính thực sự (True Positives) trong toàn bộ các trường hợp dương tính thực tế (Ground Truths).
TP (True Positive): Mô hình dự đoán đúng những trường hợp thực sự là dương tính.
FN (False Negative): Mô hình dự đoán sai những trường hợp thực sự là dương tính (bỏ sót).

In [ ]:
Image(filename=f'{HOME}/runs/detect/train4/BoxPR_curve.png', width=1000)

In [ ]:
Image(filename=f'{HOME}/runs/detect/train4/train_batch0.jpg', width=1000)

In [ ]:
Image(filename=f'{HOME}/runs/detect/train4/val_batch0_pred.jpg', width=1000)

In [ ]:
Image(filename=f'{HOME}/runs/detect/train4/val_batch1_pred.jpg', width=1000)

**Bước # 06 Chạy mô hình**
Hãy copy file best.pt từ trong thư mục run/train(số cuối cùng ra thư mục mẹ, chạy đoạn code bên dưới, lưu ý hãy xác định danh sách camera hoạt động để chọn nguồn phù hợp, trong này có cả input video, ảnh, có record lại được, câu lệnh tự tìm. Nếu đã đổi tên model rồi thì tham số model cũng phải đổi tên

In [ ]:
!python yolo_detect.py --model best.pt --source usb3 --resolution 1280x720